# VISCERA: optional text → bounded edit proposals with LoRA

This is a **separate language experiment**, not a text-to-mesh foundation model.
It prepares an original authored-template dataset, compares the deterministic
compiler and frozen small language model, trains a real LoRA adapter with loss
only on assistant JSON, then evaluates a sealed wording-family test split.
Targets include supported dimensional edits, ambiguous requests and unsupported
requests. The template set currently trains length controls; the output validator
recognizes the whole bounded pilot schema. Other capabilities need additional
authored training data before claiming coverage. State groups and wording families
are disjoint; operation families are shared. Human language generalization and
anatomical correctness are **not established**. This adapter is not deployed.

This notebook downloads `Qwen/Qwen2.5-0.5B-Instruct` at an immutable revision
(Apache-2.0). Source: [official model/license](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).
LoRA uses [official PEFT](https://huggingface.co/docs/peft/package_reference/lora).
Use a Colab GPU when available. No weights were downloaded or pretrained-model
training executed as part of the local repository verification.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

SOURCE_MODE = "upload"  # or "existing"
EXISTING_PROJECT = "/content/viscera"
INSTALL_DEPENDENCIES = True
RUN_ID = "language-lora-pilot-v0"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "c89bee90d9f811437d9735454613c35b4a3c4dc8"
TRAIN_STEPS = 200
USE_DRIVE = False
DOWNLOAD_RESULTS = True
try:
    from google.colab import drive, files

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
if SOURCE_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Outside Colab choose SOURCE_MODE='existing' and set EXISTING_PROJECT")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one VISCERA source ZIP")
    source_root = Path(tempfile.mkdtemp(prefix="viscera-source-", dir="/content"))
    with zipfile.ZipFile(archives[0]) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or "MANIFEST.json" not in names:
            raise ValueError("Source archive requires unique paths and MANIFEST.json")
        total = sum(info.file_size for info in archive.infolist())
        if total > 2_000_000_000:
            raise ValueError("Source archive exceeds the 2 GB expanded-size limit")
        for info in archive.infolist():
            target = (source_root / info.filename).resolve()
            if (
                not target.is_relative_to(source_root.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Unsafe archive path or symlink")
        manifest = json.loads(archive.read("MANIFEST.json"))
        if set(manifest["files"]) != set(names) - {"MANIFEST.json"}:
            raise ValueError("Source archive contents disagree with their manifest")
        for name, expected in manifest["files"].items():
            with archive.open(name) as source:
                if hashlib.file_digest(source, "sha256").hexdigest() != expected:
                    raise ValueError("Source checksum mismatch: " + name)
        archive.extractall(source_root)
    candidates = [
        p.parent
        for p in source_root.rglob("pyproject.toml")
        if (p.parent / "py/viscera_core/geometry.py").exists()
    ]
    if len(candidates) != 1:
        raise ValueError("Archive must contain exactly one VISCERA project")
    PROJECT = candidates[0].resolve()
elif SOURCE_MODE == "existing":
    PROJECT = Path(EXISTING_PROJECT).expanduser().resolve()
else:
    raise ValueError("SOURCE_MODE must be upload or existing")
if not (PROJECT / "py/viscera_core/geometry.py").exists():
    raise FileNotFoundError("Project geometry source is missing")
os.chdir(PROJECT)
if INSTALL_DEPENDENCIES:
    requirements = PROJECT / "notebooks/requirements-colab.txt"
    if not requirements.exists():
        raise FileNotFoundError("Pinned Colab requirements missing from source bundle")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--require-hashes", "-r", str(requirements)],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PROJECT)], check=True
    )
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / "py"))
# Child training/evaluation processes need the same source roots when using an
# existing environment without an editable package installation.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(PROJECT), str(PROJECT / "py")]
    + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
)
print("Project:", PROJECT)

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--require-hashes",
            "-r",
            str(PROJECT / "notebooks/requirements-language-colab.txt"),
        ],
        check=True,
    )
import torch

from pipelines.train.viscera_train.language import (
    audit_rows,
    compiler_predictions,
    prepare_data,
    read_rows,
    score_predictions,
)

WORK = (Path("/content/viscera-work") if IN_COLAB else PROJECT / "artifacts/notebook") / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
DATA_DIR, MODEL_DIR = WORK / "language-data", WORK / "language-model"
REMOTE = None
if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive staging requires Colab; leave USE_DRIVE=False locally")
    drive.mount("/content/drive")
    REMOTE = Path("/content/drive/MyDrive/viscera-checkpoints") / RUN_ID
    REMOTE.mkdir(parents=True, exist_ok=True)
    if (REMOTE / "checkpoint.pt").exists() and not (MODEL_DIR / "checkpoint.pt").exists():
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        for name in ("checkpoint.pt", "checkpoint.pt.sha256.json"):
            shutil.copy2(REMOTE / name, MODEL_DIR / name)
hardware = {
    "torch": torch.__version__,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vram_bytes": torch.cuda.get_device_properties(0).total_memory
    if torch.cuda.is_available()
    else None,
}
(WORK / "hardware.json").write_text(json.dumps(hardware, indent=2))
print(hardware)

In [ ]:
prepare_data(DATA_DIR)
train_rows, val_rows = read_rows(DATA_DIR, "train"), read_rows(DATA_DIR, "val")
audit_rows(train_rows, val_rows)
baseline = score_predictions(val_rows, compiler_predictions(val_rows))
(WORK / "compiler_validation.json").write_text(json.dumps(baseline, indent=2))
print(
    "Train:",
    len(train_rows),
    "Validation:",
    len(val_rows),
    "Compiler validation:",
    baseline["exact_target_rate"],
)
print("Targets are independently authored operations. Test wording is not consumed by training.")

## Train a bounded compiler adapter

The trainer downloads safe pretrained tensors with remote custom code disabled,
uses rank-8 LoRA on attention query/value projections, masks all prompt tokens,
and keeps optimizer plus RNG state for local checkpoint resume. Every 25 steps
it writes an atomic checkpoint with a checksum. Drive staging is optional.
The same device and package environment are required for comparable resumed runs;
cross-device bitwise equivalence is not claimed. The final fixed-step model is
evaluated on validation; there is no test-driven checkpoint selection.

In [ ]:
command = [
    sys.executable,
    "-m",
    "pipelines.train.viscera_train.language",
    "train",
    "--data",
    str(DATA_DIR),
    "--out",
    str(MODEL_DIR),
    "--model",
    MODEL_ID,
    "--revision",
    MODEL_REVISION,
    "--steps",
    str(TRAIN_STEPS),
]
if (MODEL_DIR / "checkpoint.pt").exists():
    command.append("--resume")


def persist_language_checkpoint():
    if REMOTE is None:
        return
    try:
        payload = (MODEL_DIR / "checkpoint.pt").read_bytes()
        sidecar = (MODEL_DIR / "checkpoint.pt.sha256.json").read_text()
        if hashlib.sha256(payload).hexdigest() != json.loads(sidecar)["sha256"]:
            return
        temporary = REMOTE / "checkpoint.pt.uploading"
        temporary.write_bytes(payload)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != json.loads(sidecar)["sha256"]:
            raise IOError("Drive checkpoint read-back mismatch")
        temporary.replace(REMOTE / "checkpoint.pt")
        (REMOTE / "checkpoint.pt.sha256.json").write_text(sidecar)
    except FileNotFoundError:
        return


process = subprocess.Popen(command, cwd=PROJECT)
try:
    while process.poll() is None:
        persist_language_checkpoint()
        time.sleep(3)
except BaseException:
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
finally:
    persist_language_checkpoint()
if process.returncode != 0:
    raise RuntimeError("Language training failed; checkpoint and stderr are retained")
report = json.loads((MODEL_DIR / "training_report.json").read_text())
print({k: report[k] for k in ("status", "step", "trainable_parameters", "wall_time_s")})

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pipelines.train.viscera_train.language",
        "evaluate",
        "--data",
        str(DATA_DIR),
        "--out",
        str(MODEL_DIR),
    ],
    check=True,
)
evaluation = json.loads((MODEL_DIR / "test_evaluation.json").read_text())
for method in ("compiler", "frozen", "adapter"):
    item = evaluation[method]
    print(
        method,
        {
            k: item[k]
            for k in ("attempted", "valid_json_schema", "exact_target_rate", "unsafe_false_ready")
        },
    )
print(
    "All invalid outputs and false-ready cases count as failures. No automatic runtime promotion."
)

In [ ]:
archive = Path(shutil.make_archive(str(WORK.parent / (RUN_ID + "-results")), "zip", WORK))
if REMOTE is not None:
    shutil.copy2(archive, REMOTE / archive.name)
    if (
        hashlib.sha256((REMOTE / archive.name).read_bytes()).hexdigest()
        != hashlib.sha256(archive.read_bytes()).hexdigest()
    ):
        raise IOError("Drive result bundle checksum mismatch")
if DOWNLOAD_RESULTS and IN_COLAB:
    files.download(str(archive))
print(
    "Language dataset, LoRA adapter, safe checkpoint, model card and three-way evaluation:", archive
)